### Describe/introduce tutorial ###
- Objectives
- Inputs and outputs
    - hist (tas)
    - hist nat (tas)
    - obs (mean temp, 0.5 degree)

#### Imports

In [1]:
# I've commented out some of the imports that aren't used in this script so far, but feel free to uncomment them if you need them
# Just wanted to keep track fo what I'm using so we can neaten up and not have a bunch of imports that we dont need in the final version of the script
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import xesmf as xe
import bias_correction_functions as bcf
import downscaling_functions as dsf
import analysis_functions as af
# from glob import glob
# from collections import defaultdict
# import functions as func
# import subprocess
# import tempfile
# import os
# import pandas as pd
# import geopandas as gpd
# from exactextract import exact_extract
# import concurrent.futures
# import gc
# from osgeo import osr

#### Load data 

In [2]:
obs = xr.open_dataarray("obs_05x05_2000_2018_padded_interp_for_tutorial.nc")
hist = xr.open_dataset("tas_CanESM5_historical_r1i1p1f1_v20190429_2000-2018.nc") # Gloabl -> could crop and save a version that is just Brazil

#### Process model data and regrid obs to model grid

In [3]:
hist_processed = af.process_model_data(hist.tas, obs)

Longitudes in 'lon' shifted to [-180, 180] range.


In [4]:
obs_processed = af.regrid_obs_to_model(obs, hist_processed)

#### Run bias correction

In [5]:
# Replace nan with zeros
obs_processed = obs_processed.fillna(0)

In [6]:
# Extract arrays as the bias correction function is designed to work with Numpy arrays. 
# Xarray has other metadata attached that will get in the way of the bias correction.
# The function is designed to work with multiple variables, so we pass in lists of arrays even though we only have one variable.

obs_arrays = [obs_processed.values]
train_arrays = [hist_processed.values]
apply_arrays = [hist_processed.values]

# Compute time metadata to pass into bias correction function
years = {
    'obs_hist': obs_processed.time.dt.year.values,
    'sim_hist': hist_processed.time.dt.year.values, 
    'sim_fut': hist_processed.time.dt.year.values
}
days = {
    'obs_hist': obs_processed.time.dt.dayofyear.values,
    'sim_hist': hist_processed.time.dt.dayofyear.values,
    'sim_fut': hist_processed.time.dt.dayofyear.values
}
month_numbers = {
    'obs_hist': obs_processed.time.dt.month.values,
    'sim_hist': hist_processed.time.dt.month.values,
    'sim_fut': hist_processed.time.dt.month.values
}

# Call the bias correction
bias_corrected = bcf.main(
    obs_arrays, train_arrays, apply_arrays,
    variable=['tas'],
    years=years, days=days, month_numbers=month_numbers,
    distribution=['normal'],
    trend_preservation=["additive"],
    detrend=[True],
    halfwin_upper_bound_climatology=[0]
)

adjusting at locations in 23x23 ...
(0, 0)
(0, 1)
(0, 2)
(0, 3)
(0, 4)
(0, 5)
(0, 6)
(0, 7)
(0, 8)
(0, 9)
(0, 10)
(0, 11)


/Users/al18709/Documents/coding_tutorials/attribution_tutorial/utility_functions.py:1572: UserWarning: found fewer then 2 different values in x: returning None
  warnings.warn(msg)
/Users/al18709/Documents/coding_tutorials/attribution_tutorial/bias_correction_functions.py:244: UserWarning: unable to do parametric quantile mapping: doing non-parametric quantile mapping instead
  if spsdotwhat is not None: warnings.warn(msg)


(0, 12)
(0, 13)
(0, 14)
(0, 15)
(0, 16)
(0, 17)
(0, 18)
(0, 19)
(0, 20)
(0, 21)
(0, 22)
(1, 0)
(1, 1)
(1, 2)
(1, 3)
(1, 4)
(1, 5)
(1, 6)
(1, 7)
(1, 8)
(1, 9)
(1, 10)
(1, 11)
(1, 12)
(1, 13)
(1, 14)
(1, 15)
(1, 16)
(1, 17)
(1, 18)
(1, 19)
(1, 20)
(1, 21)
(1, 22)
(2, 0)
(2, 1)
(2, 2)
(2, 3)
(2, 4)
(2, 5)
(2, 6)
(2, 7)
(2, 8)
(2, 9)
(2, 10)
(2, 11)
(2, 12)
(2, 13)
(2, 14)
(2, 15)
(2, 16)
(2, 17)
(2, 18)
(2, 19)
(2, 20)
(2, 21)
(2, 22)
(3, 0)
(3, 1)
(3, 2)
(3, 3)
(3, 4)
(3, 5)
(3, 6)
(3, 7)
(3, 8)
(3, 9)
(3, 10)
(3, 11)
(3, 12)
(3, 13)
(3, 14)
(3, 15)
(3, 16)
(3, 17)
(3, 18)
(3, 19)
(3, 20)
(3, 21)
(3, 22)
(4, 0)
(4, 1)
(4, 2)
(4, 3)
(4, 4)
(4, 5)
(4, 6)
(4, 7)
(4, 8)
(4, 9)
(4, 10)
(4, 11)
(4, 12)
(4, 13)
(4, 14)
(4, 15)
(4, 16)
(4, 17)
(4, 18)
(4, 19)
(4, 20)
(4, 21)
(4, 22)
(5, 0)
(5, 1)
(5, 2)
(5, 3)
(5, 4)
(5, 5)
(5, 6)
(5, 7)
(5, 8)
(5, 9)
(5, 10)
(5, 11)
(5, 12)
(5, 13)
(5, 14)
(5, 15)
(5, 16)
(5, 17)
(5, 18)
(5, 19)
(5, 20)
(5, 21)
(5, 22)
(6, 0)
(6, 1)
(6, 2)
(6, 3)
(6, 4)
(6, 5)


In [7]:
# Reconstruct the full spatial array from bias_corrected dictionary

# Get spatial dimensions from original sim array
lat_size, lon_size = hist_processed.shape[:-1]  
time_size = hist_processed.shape[-1]

# Create empty array to hold bias-corrected data
bias_corrected_array = np.full((lat_size, lon_size, time_size), np.nan)

# Fill the array with bias-corrected data
for loc, data in bias_corrected.items():
    lat_idx, lon_idx = loc
    tas_adjusted = data[0]  # First (and only) variable
    bias_corrected_array[lat_idx, lon_idx, :] = tas_adjusted

# Create xarray DataArray with proper dimensions and coordinates
tas_bias_corrected = xr.DataArray(
    bias_corrected_array,
    dims=['lat', 'lon', 'time'],
    coords={
        'lon': hist_processed.lon.values,
        'lat': hist_processed.lat.values,
        'time': hist_processed.time.values
    },
    attrs=hist_processed.attrs  # Copy attributes from original
)

#### Compare bias corrected data with raw model output and observations

In [ ]:
# # Landmask 
# obs = xr.open_dataarray('/user/work/nh25161/brazil_mask_01x01.nc')
# # Use obs to make mask
# mask_1x1 = obs.coarsen(lat=10, lon=10, boundary='trim').mean()
# mask_05x05 = obs.coarsen(lat=5, lon=5, boundary='trim').mean()
# mask_2x2 = obs.coarsen(lat=20, lon=20, boundary='trim').mean()

#### Downscale

In [8]:
bias_trim, obs_trim = af.prepare_bias_corrected_and_obs(tas_bias_corrected, obs)

Downscaling factor for lat: 4
Do actual and expected match? False
Max absolute difference: 0.25
Downscaling factor for lon: 4
Do actual and expected match? False
Max absolute difference: 0.25


In [9]:
# Replace nan with zeros
obs_trim = obs_trim.fillna(0)

In [10]:
# Extract arrays
obs_trim_arrays = [obs_trim.values]
bias_trim_arrays = [bias_trim.values]

In [11]:
# Run downscaling
sim_downscaled = dsf.main(
    obs_trim_arrays, bias_trim_arrays,
    variable=['tas'],
    years=years, days=days, month_numbers=month_numbers,
)

preparing downscaling ...
downscaling with fine resolution shape (84x84) ...
(0, 0)
(0, 1)
(0, 2)
(0, 3)
(0, 4)
(0, 5)
(0, 6)
(0, 7)
(0, 8)
(0, 9)
(0, 10)
(0, 11)
(0, 12)
(0, 13)
(0, 14)
(0, 15)
(0, 16)
(0, 17)
(0, 18)
(0, 19)
(0, 20)
(1, 0)
(1, 1)
(1, 2)
(1, 3)
(1, 4)
(1, 5)
(1, 6)
(1, 7)
(1, 8)
(1, 9)
(1, 10)
(1, 11)
(1, 12)
(1, 13)
(1, 14)
(1, 15)
(1, 16)
(1, 17)
(1, 18)
(1, 19)
(1, 20)
(2, 0)
(2, 1)
(2, 2)
(2, 3)
(2, 4)
(2, 5)
(2, 6)
(2, 7)
(2, 8)
(2, 9)
(2, 10)
(2, 11)
(2, 12)
(2, 13)
(2, 14)
(2, 15)
(2, 16)
(2, 17)
(2, 18)
(2, 19)
(2, 20)
(3, 0)
(3, 1)
(3, 2)
(3, 3)
(3, 4)
(3, 5)
(3, 6)
(3, 7)
(3, 8)
(3, 9)
(3, 10)
(3, 11)
(3, 12)
(3, 13)
(3, 14)
(3, 15)
(3, 16)
(3, 17)
(3, 18)
(3, 19)
(3, 20)
(4, 0)
(4, 1)
(4, 2)
(4, 3)
(4, 4)
(4, 5)
(4, 6)
(4, 7)
(4, 8)
(4, 9)
(4, 10)
(4, 11)
(4, 12)
(4, 13)
(4, 14)
(4, 15)
(4, 16)
(4, 17)
(4, 18)
(4, 19)
(4, 20)
(5, 0)
(5, 1)
(5, 2)
(5, 3)
(5, 4)
(5, 5)
(5, 6)
(5, 7)
(5, 8)
(5, 9)
(5, 10)
(5, 11)
(5, 12)
(5, 13)
(5, 14)
(5, 15)
(5, 16)
(5, 17)

In [12]:
# Convert downscaled array back to xarray DataArray
tas_downscaled = xr.DataArray(
    sim_downscaled[0],  # Take the array from the list
    dims=obs_trim.dims,  # Same dimensions as original
    coords=obs_trim.coords,  # Same coordinates as original
    attrs=obs_trim.attrs,  # Same attributes as original
    name='tas'  # Same variable name
)

#### Plot downscaled data 

In [22]:
# saving downscaled output for next worksheet
tas_downscaled.to_netcdf("tas_downscaled_05x05_2000_2018.nc")

In [ ]:
# Add plotting code here